### Bayesian Lasso x Bilby

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import bilby
from bilby.core.utils import random
import json

### Questions

- Do I sample $\tau^2$ or $\tau$?
- Matching exact parameter distributions?

### To Do
- Create custom inverse Gamma likelihood for $\sigma_{noise}$
- code log likelihood function 

In [2]:
# set up
random.seed(123)
label = "BLxBilby"
outdir = "outdir"
bilby.utils.check_directory_exists_and_if_not_mkdir(outdir)

sample_dat_path = "../synthetic_data/N11000_AP10_noise0.5_seed1/Size500/Rep3.csv"
sample_dat = pd.read_csv(sample_dat_path)

val_dat_path = "../synthetic_data/N11000_AP10_noise0.5_seed1/N11000_AP10_noise0.5_seed1_meta.json"
with open(val_dat_path, 'r') as f:
    val_dat = json.load(f)

true_betas = np.array(val_dat['beta'])
true_ells = np.array(val_dat['lengthscales'])
active_dims = np.array(val_dat['active_indices'])
true_sigma_noise = val_dat['noise_constant']

from sklearn.model_selection import train_test_split

X = sample_dat.iloc[:, :-1].values  #all columns except the last
y = sample_dat.iloc[:, -1].values   #last column

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=22)

### Derivation of the Log Likelihood

Part 1: Data
$$ y \mid X, \beta, \sigma_{noise}^2 \sim \mathcal{N_n}(X\beta, \sigma_{noise}^2I)$$

$$\mathcal{L}(y \mid X, \beta, \sigma_{noise}^2) = \frac{1}{(2\pi\sigma_{noise}^2)^{\frac{n}{2}}} \exp\left(-\frac{1}{2\sigma_{noise}^2} \| y - X \beta \|^2 \right)$$

$$\log \mathcal{L}(y \mid X, \beta, \sigma_{noise}^2) = -\frac{n}{2} \log(2\pi) - \frac{n}{2} \log(\sigma_{noise}^2) - \frac{1}{2\sigma_{noise}^2} \| y - X\beta \|^2$$

Part 2: Coefficients

$$ \beta_j \mid \tau_j^2, \sigma_{noise}^2 \sim \mathcal{N}(0, \tau^2_j\sigma_{noise}^2)$$

$$\mathcal{L}(\beta | \tau, \sigma_{noise}^2) = 
\prod_{j=1}^p \frac{1}{\sqrt{2\pi\sigma_{noise}^2\tau^2_j}} 
\exp\left( \frac{-\beta_j^2} {2\sigma_{noise}^2\tau^2_j}\right)$$

$$\log \mathcal{L}(\beta | \tau, \sigma_{noise}^2) = -\frac{1}{2}\sum_{j=1}^p \left( \log(2\pi\sigma_{noise}^2\tau_j^2) + \frac{\beta^2_j}{\sigma^2_{noise}\tau_j^2} \right)$$

Part 3: Scaling

$$ \tau_j^2 \sim \mathcal{Exp}(\frac{\lambda^2}{2})$$
$$\mathcal{L}(\tau_{1}^2, ..., \tau_p^2 \mid \lambda) = \prod_{j=1}^p \frac{\lambda^2}{2} \exp(\frac{-\lambda^2\tau_j^2}{2}) $$

$$\log \mathcal{L}(\tau_{1}^2, ..., \tau_p^2 \mid \lambda) = \sum^p_{j=1} \left( log(\frac{\lambda^2}{2}) - \frac{\lambda^2\tau^2_j}{2} \right)$$

#### Full Log Likelihood

$$\log \mathcal{L}(y, \beta, \tau \mid \sigma_{noise}^2, \lambda) 
= -\frac{n}{2} \log(2\pi) 
    - \frac{n}{2} \log(\sigma_{noise}^2) 
        - \frac{1}{2\sigma_{noise}^2} \| y - X\beta \|^2 
-\frac{1}{2}\sum_{j=1}^p 
    \left( \log(2\pi\sigma_{noise}^2\tau_j^2) + \frac{\beta^2_j}{\sigma^2_{noise}\tau_j^2} \right) 
+ \sum^p_{j=1} \left( \log(\frac{\lambda^2}{2}) - \frac{\lambda^2\tau^2_j}{2} \right)$$


In [3]:
# custom likelihood for multi-dimensional linear regression
class BayesianLassoLikelihood(bilby.Likelihood):
    def __init__(self, X, y):
        # store data
        self.X = np.asarray(X)
        self.y = np.asarray(y)

        # define parameters
        parameters = {}
        for i in range(self.X.shape[1]):
            parameters[f"beta{i}"] = None
            parameters[f"tau{i}"] = None
        parameters["sigma_noise"] = None
        parameters["lbda"] = None
        
        super().__init__(parameters=parameters)


    def log_likelihood(self):

        beta = np.array([self.parameters[f"beta{i}"] for i in range(self.X.shape[1])])
        tau = np.array([self.parameters[f"tau{i}"] for i in range(self.X.shape[1])])
        sigma_2 = self.parameters["sigma_noise"]**2
        lbda = self.parameters["lbda"]

        # calculate n
        n = self.X.shape[0]

        # calculate residuals
        residuals = self.y - self.X @ beta

        # calculate log likelihood
        log_likelihood_data = -0.5 * n * np.log(2 * np.pi * sigma_2) - 0.5 * np.sum(residuals**2) / sigma_2
        
        log_likelihood_beta = log_likelihood_beta = -0.5 * np.sum(
            np.log(2 * np.pi * sigma_2 * tau**2) + (beta**2) / (sigma_2 * tau**2)
        )

        log_likelihood_tau = log_likelihood_tau = np.sum(
            np.log(lbda**2 / 2) - 0.5 * lbda**2 * tau**2)

        log_likelihood = log_likelihood_data + log_likelihood_beta + log_likelihood_tau
        return log_likelihood

In [4]:
# # define custom likelihoods


# class InverseGammaPrior(bilby.prior.Prior):
#     def __init__(self, shape, scale, name):
#         super().__init__(name=name)
#         self.shape = shape
#         self.scale = scale

#     def log_likelihood(self, x):
#         return (self.shape - 1) * np.log(x) - x / self.scale - self.shape * np.log(self.scale) - np.log(bilby.core.utils.gamma(self.shape))

In [5]:
# model function

def model_function(X, **params): 
    betas = np.array([params[f"beta{i}"] for i in range(X.shape[1])]) # make beta for each column
    return X @ betas 

# make priors
priors = dict()

for i in range(30):
    priors[f"beta{i}"] = bilby.core.prior.Normal(0, 2, f"beta{i}") # define normal priors for each beta coefficient 
    priors[f"tau{i}"] = bilby.core.prior.Exponential(1, f"tau{i}") # define exp priors for each tau

# priors["sigma_noise"] = InverseGammaPrior(1, 1, "sigma_noise")  ### edit shape and scale 
priors["sigma_noise"] = bilby.core.prior.LogNormal(0, 1, "sigma_noise")  
priors["lbda"] = bilby.core.prior.LogNormal(0, 1, "lbda")  ### edit distribution

# define the likelihood function that we defined earlier
likelihood = BayesianLassoLikelihood(
    X = Xtrain,
    y = ytrain)

# debugging
print("Prior keys:")
print(sorted(priors.keys()))
print("\nLikelihood parameter keys:")
print(sorted(likelihood.parameters.keys()))
print("\nNumber of parameters:")
print(f"len(priors) = {len(priors)}, len(likelihood.parameters) = {len(likelihood.parameters)}")

assert set(priors.keys()) == set(likelihood.parameters.keys()), "Mismatch between priors and likelihood parameters!"


# run MCMC sampler
result = bilby.run_sampler(
    likelihood=likelihood, # likelihood function
    priors=priors, # prior distributions
    sampler="emcee", 
    nwalkers = 200,
    nsteps = 50, 
    nburn = 10,
    outdir=outdir,
    label=label
)

08:58 bilby INFO    : Running for label 'BLxBilby', output will be saved to 'outdir'
08:58 bilby INFO    : Analysis priors:
08:58 bilby INFO    : beta0=Normal(mu=0, sigma=2, name='beta0', latex_label='beta0', unit=None, boundary=None)
08:58 bilby INFO    : tau0=Exponential(mu=1, name='tau0', latex_label='tau0', unit=None, boundary=None)
08:58 bilby INFO    : beta1=Normal(mu=0, sigma=2, name='beta1', latex_label='beta1', unit=None, boundary=None)
08:58 bilby INFO    : tau1=Exponential(mu=1, name='tau1', latex_label='tau1', unit=None, boundary=None)
08:58 bilby INFO    : beta2=Normal(mu=0, sigma=2, name='beta2', latex_label='beta2', unit=None, boundary=None)
08:58 bilby INFO    : tau2=Exponential(mu=1, name='tau2', latex_label='tau2', unit=None, boundary=None)
08:58 bilby INFO    : beta3=Normal(mu=0, sigma=2, name='beta3', latex_label='beta3', unit=None, boundary=None)
08:58 bilby INFO    : tau3=Exponential(mu=1, name='tau3', latex_label='tau3', unit=None, boundary=None)
08:58 bilby INFO

Prior keys:
['beta0', 'beta1', 'beta10', 'beta11', 'beta12', 'beta13', 'beta14', 'beta15', 'beta16', 'beta17', 'beta18', 'beta19', 'beta2', 'beta20', 'beta21', 'beta22', 'beta23', 'beta24', 'beta25', 'beta26', 'beta27', 'beta28', 'beta29', 'beta3', 'beta4', 'beta5', 'beta6', 'beta7', 'beta8', 'beta9', 'lbda', 'sigma_noise', 'tau0', 'tau1', 'tau10', 'tau11', 'tau12', 'tau13', 'tau14', 'tau15', 'tau16', 'tau17', 'tau18', 'tau19', 'tau2', 'tau20', 'tau21', 'tau22', 'tau23', 'tau24', 'tau25', 'tau26', 'tau27', 'tau28', 'tau29', 'tau3', 'tau4', 'tau5', 'tau6', 'tau7', 'tau8', 'tau9']

Likelihood parameter keys:
['beta0', 'beta1', 'beta10', 'beta11', 'beta12', 'beta13', 'beta14', 'beta15', 'beta16', 'beta17', 'beta18', 'beta19', 'beta2', 'beta20', 'beta21', 'beta22', 'beta23', 'beta24', 'beta25', 'beta26', 'beta27', 'beta28', 'beta29', 'beta3', 'beta4', 'beta5', 'beta6', 'beta7', 'beta8', 'beta9', 'lbda', 'sigma_noise', 'tau0', 'tau1', 'tau10', 'tau11', 'tau12', 'tau13', 'tau14', 'tau15', 't

08:58 bilby INFO    : Single likelihood evaluation took 3.438e-04 s
08:58 bilby INFO    : Global meta data was removed from the result object for compatibility. Use the `BILBY_INCLUDE_GLOBAL_METADATA` environment variable to include it. This behaviour will be removed in a future release. For more details see: https://bilby-dev.github.io/bilby/faq.html#global-meta-data
08:58 bilby WARNING : Using cached result
08:58 bilby INFO    : Summary of results:
nsamples: 8000
ln_noise_evidence:    nan
ln_evidence:    nan +/-    nan
ln_bayes_factor:    nan +/-    nan

